# Module 13 — Files, Data Formats, and the Standard Library

**Part III: Intermediate** · Level: Intermediate · Prerequisites: Modules 9, 12

## What you'll learn

- Reading and writing files safely with `with`
- Working with paths using `pathlib`
- Saving data as JSON, CSV, and pickle
- A tour of the most useful standard-library modules

---

## 13.1 Reading and writing files

`open()` returns a file object. Always use it with **`with`**, which closes the file even if an error occurs.

| Mode | Meaning |
|---|---|
| `"r"` | read text (default) |
| `"w"` | write text, **erasing** existing content |
| `"a"` | append |
| `"x"` | create, fail if it exists |
| `"rb"` / `"wb"` | binary read / write |

In [ ]:
import tempfile, os
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
path = workdir / "demo.txt"

with open(path, "w", encoding="utf-8") as f:
    f.write("hello\n")
    f.write("second line\n")

with open(path, "a", encoding="utf-8") as f:
    f.write("appended\n")

with open(path, encoding="utf-8") as f:
    print(f.read())

with open(path, encoding="utf-8") as f:
    for line in f:                       # memory-friendly: one line at a time
        print(line.rstrip("\n"))

with open(path, encoding="utf-8") as f:
    lines = f.readlines()                # list of all lines
print(len(lines))

os.remove(path)

Always pass `encoding="utf-8"` for text files so behaviour does not depend on the operating system.

## 13.2 `pathlib`

`pathlib.Path` treats paths as objects and works the same on Windows, macOS, and Linux.

In [ ]:
from pathlib import Path
import tempfile

base = Path(tempfile.mkdtemp())
notes = base / "notes" / "todo.txt"        # `/` joins path parts
notes.parent.mkdir(parents=True, exist_ok=True)

notes.write_text("buy milk\nwrite code\n", encoding="utf-8")
print(notes.read_text(encoding="utf-8"))

print(notes.name, notes.stem, notes.suffix)     # todo.txt todo .txt
print(notes.exists(), notes.is_file(), notes.parent.is_dir())
print([p.name for p in base.rglob("*.txt")])    # recursive search

notes.unlink()                                  # delete the file

## 13.3 JSON

`json` converts between Python objects (dicts, lists, strings, numbers, booleans, `None`) and text. It is the universal format for configuration and web APIs.

In [ ]:
import json

config = {"retries": 3, "hosts": ["a.example", "b.example"], "debug": False}

text = json.dumps(config, indent=2)       # object -> string
print(text)
print(json.loads(text) == config)         # string -> object

from pathlib import Path
import tempfile
file = Path(tempfile.mkdtemp()) / "config.json"
with open(file, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)        # object -> file
with open(file, encoding="utf-8") as f:
    print(json.load(f)["hosts"])          # file -> object

# JSON has no tuples/sets/dates: tuples come back as lists
print(json.loads(json.dumps({"point": (1, 2)})))

## 13.4 CSV

In [ ]:
import csv, io

rows = [
    {"date": "2026-10-01", "category": "food", "amount": "12.50"},
    {"date": "2026-10-02", "category": "rent", "amount": "900.00"},
]

buffer = io.StringIO()                    # stands in for a real file
writer = csv.DictWriter(buffer, fieldnames=["date", "category", "amount"])
writer.writeheader()
writer.writerows(rows)
print(buffer.getvalue())

buffer.seek(0)
for record in csv.DictReader(buffer):
    print(record["category"], float(record["amount"]))

With a real file, open it with `newline=""`: `open(path, "w", newline="", encoding="utf-8")`.

## 13.5 Pickle

`pickle` serialises almost any Python object to bytes.

In [ ]:
import pickle

obj = {"shots": 1024, "tags": {"a", "b"}}
blob = pickle.dumps(obj)
print(pickle.loads(blob) == obj)          # True; sets survive, unlike JSON

> **Security warning:** never unpickle data from an untrusted source — loading a pickle can execute arbitrary code. For data you exchange with others, use JSON.

## 13.6 Standard-library tour

Python's "batteries included" library saves you from reinventing common tools. Browse it at docs.python.org/3/library.

### `math` and `statistics`

In [ ]:
import math
from statistics import mean, median, stdev

print(math.sqrt(9), math.factorial(5), math.pi, math.sin(0), math.ceil(2.1), math.floor(2.9))
print(math.gcd(12, 18), math.log2(1024), math.hypot(3, 4))

data = [1, 2, 3, 4, 10]
print(mean(data), median(data), round(stdev(data), 3))

### `random`

In [ ]:
import random

random.seed(42)                       # reproducible results for tests and demos
print(random.randint(1, 6))
print(random.choice(["red", "green", "blue"]))
print(random.sample(range(10), 3))
deck = list(range(5))
random.shuffle(deck)
print(deck)

`random` is **not** suitable for passwords or security tokens; use the `secrets` module.

### `os` and `sys`

In [ ]:
import os, sys

print(os.getcwd() != "")
print(os.environ.get("HOME") or os.environ.get("USERPROFILE"))
print(os.name, sys.platform)
print(sys.version.split()[0])
print(sys.argv[:1])                   # command-line arguments (script name first)

### `collections`

In [ ]:
from collections import Counter, defaultdict, deque

print(Counter(["a", "a", "b"]))
print(Counter("mississippi").most_common(2))

groups = defaultdict(list)            # missing keys start as an empty list
for word in ["apple", "avocado", "banana"]:
    groups[word[0]].append(word)
print(dict(groups))

queue = deque([1, 2, 3])
queue.appendleft(0)                   # O(1) at both ends, unlike a list
queue.pop()
print(queue)

### `time` and `datetime`

In [ ]:
import time
from datetime import datetime, timedelta, date

start = time.perf_counter()           # best clock for measuring durations
time.sleep(0.01)
print(f"took {time.perf_counter() - start:.2f}s")

now = datetime.now()
print(now.year >= 2026)
due = date(2026, 10, 3) + timedelta(days=30)
print(due.isoformat())
print(datetime(2026, 10, 3, 14, 30).strftime("%d %b %Y, %H:%M"))
print(datetime.strptime("2026-12-25", "%Y-%m-%d").weekday())   # 0 = Monday

### Other modules worth knowing

| Module | Use |
|---|---|
| `argparse` | command-line interfaces |
| `logging` | structured diagnostics (Module 18) |
| `shutil` | copy, move, and delete files and folders |
| `subprocess` | run other programs |
| `re` | regular expressions (Module 7) |
| `itertools`, `functools` | iteration and function tools (Modules 11, 17) |
| `sqlite3` | an embedded SQL database |
| `urllib`, `http` | basic web access |
| `secrets` | cryptographically strong randomness |

## Common pitfalls

- Opening with mode `"w"` and wiping a file you meant to append to.
- Forgetting `encoding=` and getting garbled text on another machine.
- Building paths with string concatenation instead of `Path` objects.
- Assuming JSON round-trips tuples, sets, or dates (it does not).
- Using `random` for anything security-related.

## Practice

1. Write a function that counts the lines and words in a text file.
2. Save a list of dictionaries to JSON, read it back, and confirm it is equal.
3. Use `Counter` to find the three most common words in a paragraph.